# DescriPyTor — Feature Extraction

Demonstrates feature extraction from feather files using `feather_example/`.

**Workflow:** setup → load molecules → visualize → extract features → inspect results

> To convert Gaussian log files to feather format first: `logs_to_feather('path/to/log_files/')`

In [1]:
# ===Jupyter notebook setup===
%reload_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ROOT_DIR is the MolFeatures directory (one level above this notebook)
ROOT_DIR = str(Path().resolve().parent)
print(f'ROOT_DIR: {ROOT_DIR}')

sys.path.insert(0, ROOT_DIR)
sys.path.insert(0, os.path.join(ROOT_DIR, 'M2_data_extractor'))
sys.path.insert(0, os.path.join(ROOT_DIR, 'M3_modeler'))
# Note: do NOT add ROOT_DIR/utils directly — it breaks internal utils imports
os.chdir(ROOT_DIR)

# Clear stale module caches
for _mod in ['data_extractor', 'feather_extractor', 'gaussian_handler',
             'help_functions', 'modeling', 'plot', 'visualize']:
    sys.modules.pop(_mod, None)

try:
    from data_extractor import Molecules, extract_connectivity
    from feather_extractor import logs_to_feather
    print('Imports OK')
except ModuleNotFoundError as e:
    print(f'Import error: {e}')
    print('Check that ROOT_DIR points to the MolFeatures directory.')

pd.set_option('display.max_columns', None)

ROOT_DIR: C:\Users\edens\Documents\GitHub\DescriPyTor_to_upload\MolFeatures
C:\Users\edens\Desktop\DescriPytor\DescriPyTor-main\MolFeatures\MolAlign
Imports OK


In [6]:
# === Load molecules from feather files ===
# feather_example/ ships with the repo and contains pre-computed feather files.
# To use your own data: logs_to_feather('path/to/your/log_files/') first,
# then point feather_path at the resulting feather directory.

NOTEBOOK_DIR = Path(ROOT_DIR) / 'Getting_started_with_examples'
feather_path = str(NOTEBOOK_DIR / 'feather_example')

mols = Molecules(feather_path)


Molecules Loaded: ['basic.feather', 'm_Br.feather', 'm_Cl.feather', 'm_F.feather', 'm_I.feather', 'm_nitro.feather', 'o_Br.feather', 'o_Cl.feather', 'o_F.feather', 'o_I.feather', 'o_nitro.feather', 'penta_F.feather', 'p_amine.feather', 'p_azide.feather', 'p_boc.feather', 'p_Br.feather', 'p_Cl.feather', 'p_F.feather', 'p_I.feather', 'p_Me.feather', 'p_nitro.feather', 'p_OEt.feather', 'p_OH.feather', 'p_OMe.feather', 'p_Ph.feather', 'p_tfm.feather'] Failed Molecules: []


In [7]:
# === Visualize a molecule to identify atom indices ===
# Pass a list of molecule indices. Atom indices in the 3D viewer are used below.

mols.visualize_molecules([0, 1])

In [8]:
# === Extract features ===
# Loads atom-index selections from the bundled input_example.json.
# Edit that file (or create one via the GUI) to match your dataset.

input_file = str(NOTEBOOK_DIR / 'input_example.json')
with open(input_file) as f:
    answers_dict = json.load(f)

print('Feature inputs loaded:')
for k, v in answers_dict.items():
    print(f'  {k.split(chr(10))[0][:60]!r}: {v}')

# Extract — saves features_example_output.csv in ROOT_DIR
features_df = mols.get_molecules_features_set(
    entry_widgets=answers_dict,
    answers_list=None,
    save_as=True,
    csv_file_name='features_example_output'
)

Feature inputs loaded:
  'Ring Vibration atoms - by order -> Pick primary atom and par': 9
  'Stretch Threshold': 1400
  'Strech Vibration atoms- enter bonded atom pairs: ': 
  'Bend Threshold': 1600
  'Bending Vibration atoms - enter atom pairs that have a commo': 5,6
  'Dipole atoms - indices for coordination transformation: ': 1,2,19 20,19,27
  'Sub-Atoms': 1,3,5,11,8,12,14,4,6
  'NPA manipulation atoms - Insert atoms to show NPA: ': 1,2,19 20,19,27
  'charge values - Insert atoms to show charge: ': 3,5,11,12,4,6,20,21,22,7
  'charge difference - Insert atoms to show charge difference: ': 3,5 11,12
  'Sterimol atoms - Primary axis along: ': 1,4 4,1
  'Bond length - Atom pairs to calculate difference: ': 1,2 4,7
  'Bond Angle - Insert a list of atom triads/quartets for which': 8,7,4,1 1,2,19
[get_bend_vibration_single] molecule=basic, atom_pair=5, threshold=1600
Skipping pair 5 for basic: 'int' object is not iterable
[get_bend_vibration_single] molecule=basic, atom_pair=6, threshold=

In [9]:
# === Inspect results ===
print(f'Feature matrix shape: {features_df.shape}')
print(f'Columns: {list(features_df.columns)}')
features_df.head()

Feature matrix shape: (26, 32)
Columns: ['cross_9_0', 'cross_angle_9_0', 'para_9_0', 'para_angle_9_0', 'dip_x_NPA_1-2-19', 'dip_y_NPA_1-2-19', 'dip_z_NPA_1-2-19', 'total_dipole_NPA_1-2-19', 'dip_x_NPA_20-19-27', 'dip_y_NPA_20-19-27', 'dip_z_NPA_20-19-27', 'total_dipole_NPA_20-19-27', 'dipole_x_1-2-19', 'dipole_y_1-2-19', 'dipole_z_1-2-19', 'total_dipole_1-2-19', 'dipole_x_20-19-27', 'dipole_y_20-19-27', 'dipole_z_20-19-27', 'total_dipole_20-19-27', 'B1_1-4', 'B5_1-4', 'L_1-4', 'loc_B5_1-4', 'B1_B5_angle_1-4', 'B1_4-1', 'B5_4-1', 'L_4-1', 'loc_B5_4-1', 'B1_B5_angle_4-1', 'aniso', 'iso']


,cross_9_0,cross_angle_9_0,para_9_0,para_angle_9_0,dip_x_NPA_1-2-19,dip_y_NPA_1-2-19,dip_z_NPA_1-2-19,total_dipole_NPA_1-2-19,dip_x_NPA_20-19-27,dip_y_NPA_20-19-27,dip_z_NPA_20-19-27,total_dipole_NPA_20-19-27,dipole_x_1-2-19,dipole_y_1-2-19,dipole_z_1-2-19,total_dipole_1-2-19,dipole_x_20-19-27,dipole_y_20-19-27,dipole_z_20-19-27,total_dipole_20-19-27,B1_1-4,B5_1-4,L_1-4,loc_B5_1-4,B1_B5_angle_1-4,B1_4-1,B5_4-1,L_4-1,loc_B5_4-1,B1_B5_angle_4-1,aniso,iso
basic,1657.1445,90.0,1680.2718,90.0,5.256603,-0.153041,1.679644,5.520553,2.529466,1.275037,-6.630086,7.209851,-2.753279,3.530534,2.859004,5.3122,-4.897081,-0.616581,1.964063,5.3122,1.6991,5.8768,5.3493,3.7970,21.4479,2.0563,3.4034,7.4167,3.7781,15.8439,144.750,979.091
m_Br,1641.5631,90.0,1669.3498,90.0,5.321497,-0.227956,1.484323,5.529332,2.320815,1.340704,-6.694566,7.211163,-0.668946,3.170669,2.634257,4.1761,-3.063362,-1.637565,2.318219,4.1761,1.6991,5.9254,7.0545,5.1045,2.6647,2.0456,3.4206,7.4085,3.7593,15.3178,167.296,145.873
m_Cl,1647.6226,90.0,1672.1898,90.0,5.306822,-0.221149,1.488210,5.515980,2.337960,1.339065,-6.671513,7.195015,-0.781868,3.184321,2.625917,4.2008,-3.147125,-1.563782,2.301489,4.2008,1.6991,5.8868,6.7751,3.7831,21.9103,2.0455,3.4211,7.4074,3.7593,15.3529,158.957,131.355
m_F,1667.7992,90.0,1687.4749,90.0,5.489232,-0.303865,1.538555,5.708866,2.392443,1.314689,-6.979722,7.494578,-0.999625,3.223469,2.614498,4.2691,-3.321752,-1.428708,2.269440,4.2691,1.6991,5.8748,6.0469,3.8180,21.7178,2.0509,3.4146,7.4117,3.7660,15.8198,145.049,104.465
m_I,1635.9083,90.0,1665.1876,90.0,5.325433,-0.223012,1.539020,5.547843,2.384560,1.323705,-6.714034,7.246832,-0.933375,3.176521,2.569959,4.1911,-3.231589,-1.436747,2.249167,4.1911,1.6991,6.3463,7.2740,5.1240,4.1767,2.0471,3.4178,7.4089,3.7626,15.2995,179.167,163.638
